# Optimizing hyperparameters with grid-search

### Loading the data from disk (as saved by Prepare-training-data)

In [1]:
import xgboost as xgb
import sklearn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_validate
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import roc_curve, auc
from scipy.stats import ks_2samp

In [2]:
%matplotlib inline
plt.rcParams["figure.figsize"] = (15,8)
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

In [3]:
data = np.load("bdt_training_data.npz")
Xtrain_scaled = data['Xtrain_scaled']
ytrain = data['ytrain']
Xvalid_scaled = data['Xvalid_scaled']
yvalid = data['yvalid']
Xtest_scaled = data['Xtest_scaled']
ytest = data['ytest']
train_columns = data['train_columns']
print(f"Ratio bkg/sig:{ np.count_nonzero(ytrain == 0) / np.count_nonzero(ytrain)}")

Ratio bkg/sig:50.82315420188168


In [4]:
Xtrain_scaled.shape

(324983, 23)

In [5]:
Xvalid_scaled.shape

(162408, 23)

In [6]:
Xtest_scaled.shape

(162408, 23)

 # Utility methods

In [7]:
def get_bdt_reponse_similarity(clf, X_train, y_train, X_test, y_test):

    # Getting the BDT response
    train_sig = clf.predict_proba(X_train[y_train==1])[:,1]
    train_bkg = clf.predict_proba(X_train[y_train==0])[:,1]
    test_sig = clf.predict_proba(X_test[y_test==1])[:,1]
    test_bkg = clf.predict_proba(X_test[y_test==0])[:,1]
    
    ks_sig = ks_2samp(train_sig, test_sig, alternative='two-sided')
    ks_bkg = ks_2samp(train_bkg, test_bkg, alternative='two-sided')
    
    return ks_sig, ks_bkg

In [8]:
def train_model(Xtrain_scaled, ytrain, Xvalid_scaled, yvalid, Xtest_scaled, ytest, n_estimators, eta, max_depth):

    # Used for the scale_pos_weight in the model
    ratio = np.count_nonzero(ytrain == 0) / np.count_nonzero(ytrain)

    
    model = xgb.XGBClassifier( eval_metric='logloss', 
                              n_estimators=n_estimators, 
                              eta=eta, 
                              #gamma=0.10,
                              #reg_lambda=4.34,
                              scale_pos_weight=ratio, 
                              max_depth=max_depth, 
                              random_state=42, tree_method="gpu_hist", verbosity=1, seed=42)

    
    # Now train
    evalset = [(Xtrain_scaled, ytrain), (Xvalid_scaled, yvalid)]
    res = model.fit(Xtrain_scaled, ytrain, eval_set=evalset, verbose=False)
    
    # Predict on test set 
    yscore = model.predict_proba(Xtest_scaled)
    fpr, tpr, _ = roc_curve(ytest, yscore[:,1])
    roc_auc = auc(fpr, tpr)
    
    # Measure overtraining
    kss, ksb = get_bdt_reponse_similarity(model, Xtrain_scaled, ytrain, Xtest_scaled, ytest)
    return roc_auc, kss, ksb

In [9]:
# %%time
# roc_auc, kss, ksb  =  train_model(Xtrain_scaled, ytrain, Xvalid_scaled, yvalid, Xtest_scaled, ytest)
# print(roc_auc, kss, ksb)

In [10]:
def cross_train_model(Xtrain_scaled, ytrain, Xvalid_scaled, yvalid, n_estimators=200, eta=0.03, max_depth=4):

    # Used for the scale_pos_weight in the model
    ratio = np.count_nonzero(ytrain == 0) / np.count_nonzero(ytrain)

    
    model = xgb.XGBClassifier( eval_metric='logloss', 
                              n_estimators=n_estimators, 
                              eta=eta, 
                              #gamma=0.10,
                              #reg_lambda=4.34,
                              scale_pos_weight=ratio, 
                              max_depth=max_depth, 
                              random_state=42, tree_method="gpu_hist", verbosity=1, seed=42)

    
    cvs = cross_validate(model, Xtrain_scaled, ytrain, cv=5, scoring='roc_auc', return_estimator=True)
    all_auc = []
    all_kss = []
    all_ksb = []
    
    for i, model in enumerate(cvs['estimator']):
        # Predict on test set 
        yscore = model.predict_proba(Xvalid_scaled)
        fpr, tpr, _ = roc_curve(yvalid, yscore[:,1])
        roc_auc = auc(fpr, tpr)        
            
        # Measure overtraining
        kss, ksb = get_bdt_reponse_similarity(model, Xtrain_scaled, ytrain, Xvalid_scaled, yvalid)
        all_auc.append(roc_auc)
        all_kss.append(kss)
        all_ksb.append(ksb)
    return all_auc, all_kss, all_ksb

In [11]:
%time
import pickle

Xtrain_scaled_tmp = np.concatenate((Xtrain_scaled, Xvalid_scaled))
ytrain_tmp = np.concatenate((ytrain, yvalid))

cv_all_results = {}
single_all_results = {}

for n_estimator in range(100, 501, 50):
    for eta in  np.arange(0.01, 0.11, 0.01):
        for max_depth in range(2, 7):
    
            # Cross validation
            result = cross_train_model(Xtrain_scaled_tmp, ytrain_tmp, Xtest_scaled, ytest, n_estimator, eta, max_depth)
            #print(f"{n_estimator} {eta} {max_depth} {result}")
            cv_all_results[(n_estimator, eta, max_depth)] = result
            with open("cv_result.pkl", "wb") as f:
                pickle.dump(cv_all_results, f)

            # Single run 
            result2 =  train_model(Xtrain_scaled, ytrain, Xvalid_scaled, yvalid, Xtest_scaled, ytest, n_estimator, eta, max_depth)
            print(f"{n_estimator} {eta} {max_depth} {result2}")
            single_all_results[(n_estimator, eta, max_depth)] = result2
            with open("single_result.pkl", "wb") as f:
                pickle.dump(single_all_results, f)
            


CPU times: user 2 µs, sys: 1 µs, total: 3 µs
Wall time: 7.15 µs
100 0.01 2 (0.8303694943581672, KstestResult(statistic=0.028234318547610614, pvalue=0.07232299613131361, statistic_location=0.6277615, statistic_sign=-1), KstestResult(statistic=0.0021164534103682575, pvalue=0.7273075027233089, statistic_location=0.49241838, statistic_sign=1))
100 0.01 3 (0.8454954522248341, KstestResult(statistic=0.03160515277813482, pvalue=0.031321825597763114, statistic_location=0.6305193, statistic_sign=-1), KstestResult(statistic=0.0025851243905393173, pvalue=0.47592991301487075, statistic_location=0.31556982, statistic_sign=-1))
100 0.01 4 (0.8491338800126771, KstestResult(statistic=0.036355251479905884, pvalue=0.008210500529454434, statistic_location=0.56547254, statistic_sign=-1), KstestResult(statistic=0.0023466191574862405, pvalue=0.6014454466030441, statistic_location=0.5690275, statistic_sign=1))
100 0.01 5 (0.8548474383522482, KstestResult(statistic=0.03999023870130452, pvalue=0.00259775307026

100 0.060000000000000005 6 (0.894201890384645, KstestResult(statistic=0.1202936272302165, pvalue=1.6412211783707238e-26, statistic_location=0.65699893, statistic_sign=-1), KstestResult(statistic=0.0034118669437992466, pvalue=0.16816718678066134, statistic_location=0.6249031, statistic_sign=1))
100 0.06999999999999999 2 (0.8693476803366702, KstestResult(statistic=0.028396937894254286, pvalue=0.06961202944058527, statistic_location=0.6868968, statistic_sign=-1), KstestResult(statistic=0.0020071691980904152, pvalue=0.784716810143421, statistic_location=0.06262373, statistic_sign=-1))
100 0.06999999999999999 3 (0.8806993672981503, KstestResult(statistic=0.0351051928381488, pvalue=0.01189206861788162, statistic_location=0.6518502, statistic_sign=-1), KstestResult(statistic=0.002539907676491482, pvalue=0.4988297515886403, statistic_location=0.64767444, statistic_sign=1))
100 0.06999999999999999 4 (0.8898392560774379, KstestResult(statistic=0.052061207977265156, pvalue=2.5943994461139524e-05,

150 0.02 5 (0.8762390037560644, KstestResult(statistic=0.051232697578238964, pvalue=3.69908059076809e-05, statistic_location=0.5505646, statistic_sign=-1), KstestResult(statistic=0.002792756792268078, pvalue=0.37809111995929756, statistic_location=0.08170047, statistic_sign=-1))
150 0.02 6 (0.8802205837257702, KstestResult(statistic=0.0789597099706711, pvalue=1.1691966065385918e-11, statistic_location=0.541259, statistic_sign=-1), KstestResult(statistic=0.002738279439958413, pvalue=0.4025077166509592, statistic_location=0.6022295, statistic_sign=1))
150 0.03 2 (0.8630419403867852, KstestResult(statistic=0.03150382602874711, pvalue=0.03216394771860336, statistic_location=0.6819985, statistic_sign=-1), KstestResult(statistic=0.0021200421725690033, pvalue=0.7253759540968294, statistic_location=0.23662095, statistic_sign=-1))
150 0.03 3 (0.8714692574429741, KstestResult(statistic=0.036137546381297986, pvalue=0.008765847547360478, statistic_location=0.6475037, statistic_sign=-1), KstestResu

150 0.08 4 (0.8971052932266399, KstestResult(statistic=0.06840998675200596, pvalue=7.377946588679945e-09, statistic_location=0.666893, statistic_sign=-1), KstestResult(statistic=0.0028683733842619352, pvalue=0.3457783371518697, statistic_location=0.64957845, statistic_sign=1))
150 0.08 5 (0.8989332500133984, KstestResult(statistic=0.10938085528569515, pvalue=5.483774060816492e-22, statistic_location=0.57717985, statistic_sign=-1), KstestResult(statistic=0.003139987815463763, pvalue=0.24525951354687692, statistic_location=0.54582286, statistic_sign=1))
150 0.08 6 (0.8993324222632998, KstestResult(statistic=0.1773210355728269, pvalue=3.033055874909699e-57, statistic_location=0.5934965, statistic_sign=-1), KstestResult(statistic=0.004350880606066587, pvalue=0.035741503639137306, statistic_location=0.5135516, statistic_sign=1))
150 0.09 2 (0.883708030946115, KstestResult(statistic=0.025645495100135293, pvalue=0.12900126996349498, statistic_location=0.68893045, statistic_sign=-1), KstestRes

200 0.04 4 (0.8919043072107388, KstestResult(statistic=0.05564741973941284, pvalue=5.231552438143438e-06, statistic_location=0.6542119, statistic_sign=-1), KstestResult(statistic=0.002819853539084205, pvalue=0.36629853243448673, statistic_location=0.66660076, statistic_sign=1))
200 0.04 5 (0.8960308314975617, KstestResult(statistic=0.08886635229340088, pvalue=1.1815121091601899e-14, statistic_location=0.675392, statistic_sign=-1), KstestResult(statistic=0.0033600681311241054, pvalue=0.1811573804781682, statistic_location=0.61809796, statistic_sign=1))
200 0.04 6 (0.8970245988400118, KstestResult(statistic=0.14165189911621248, pvalue=1.269457985355934e-36, statistic_location=0.5998749, statistic_sign=-1), KstestResult(statistic=0.0035907151618703725, pvalue=0.12890767717382623, statistic_location=0.65424687, statistic_sign=1))
200 0.05 2 (0.8773547226966424, KstestResult(statistic=0.03137787879303548, pvalue=0.03323823114638214, statistic_location=0.70539236, statistic_sign=-1), KstestR

...

400 0.05 5 (0.9005682890523133, KstestResult(statistic=0.14587793318051737, pvalue=8.03589273750752e-39, statistic_location=0.59068537, statistic_sign=-1), KstestResult(statistic=0.0037313989805357606, pvalue=0.10357412711741121, statistic_location=0.53843653, statistic_sign=1))
400 0.05 6 (0.8995569066486752, KstestResult(statistic=0.23862692582245743, pvalue=9.196973063458395e-104, statistic_location=0.63709843, statistic_sign=-1), KstestResult(statistic=0.004841140403398447, pvalue=0.013716398880473962, statistic_location=0.49873742, statistic_sign=1))
400 0.060000000000000005 2 (0.8932976071458618, KstestResult(statistic=0.032435783849302346, pvalue=0.02511994914970173, statistic_location=0.69677556, statistic_sign=-1), KstestResult(statistic=0.0025529612370002175, pvalue=0.4921670829808623, statistic_location=0.6433312, statistic_sign=1))
400 0.060000000000000005 3 (0.8985842845050726, KstestResult(statistic=0.05844417250416156, pvalue=1.3936504809683228e-06, statistic_location=0.

450 0.01 4 (0.8800427515142921, KstestResult(statistic=0.040642991931152846, pvalue=0.0020884785314414494, statistic_location=0.63936704, statistic_sign=-1), KstestResult(statistic=0.002524172504199118, pvalue=0.5069145890229549, statistic_location=0.5902793, statistic_sign=1))
450 0.01 5 (0.8862449387236608, KstestResult(statistic=0.062160624570014325, pvalue=2.173430382663466e-07, statistic_location=0.4936092, statistic_sign=-1), KstestResult(statistic=0.0030454389288768757, pvalue=0.2774900824914711, statistic_location=0.5808209, statistic_sign=1))
450 0.01 6 (0.8899607994906749, KstestResult(statistic=0.0995183332882475, pvalue=2.8523153852294073e-18, statistic_location=0.6314184, statistic_sign=-1), KstestResult(statistic=0.0032039791007699714, pvalue=0.22507924154658843, statistic_location=0.6084835, statistic_sign=1))
450 0.02 2 (0.8747610680580105, KstestResult(statistic=0.03042699009434043, pvalue=0.042415012951062016, statistic_location=0.6843109, statistic_sign=-1), KstestRe

450 0.06999999999999999 4 (0.9012830452262243, KstestResult(statistic=0.11881893251260908, pvalue=7.109415838499682e-26, statistic_location=0.66225713, statistic_sign=-1), KstestResult(statistic=0.002987838638784046, pvalue=0.29856375569575266, statistic_location=0.6697255, statistic_sign=1))
450 0.06999999999999999 5 (0.8995509039895695, KstestResult(statistic=0.20790754159116515, pvalue=1.0952301581839906e-78, statistic_location=0.5679274, statistic_sign=-1), KstestResult(statistic=0.004515034558897901, pvalue=0.026232865315094056, statistic_location=0.512259, statistic_sign=1))
450 0.06999999999999999 6 (0.8968607222765479, KstestResult(statistic=0.33289256111143495, pvalue=1.7688848157462514e-203, statistic_location=0.682247, statistic_sign=-1), KstestResult(statistic=0.005363568316971379, pvalue=0.004417725760088075, statistic_location=0.56133604, statistic_sign=1))
450 0.08 2 (0.8971897355852585, KstestResult(statistic=0.03734151919676387, pvalue=0.006073743297407008, statistic_l

500 0.03 3 (0.8944016536272292, KstestResult(statistic=0.0509321310949863, pvalue=4.201398378744199e-05, statistic_location=0.6761827, statistic_sign=-1), KstestResult(statistic=0.00287841751894935, pvalue=0.3416268355967993, statistic_location=0.64934444, statistic_sign=1))
500 0.03 4 (0.898441651878447, KstestResult(statistic=0.0761111817714471, pvalue=7.303942608718231e-11, statistic_location=0.6803639, statistic_sign=-1), KstestResult(statistic=0.0034736674521822497, pvalue=0.15364670558614313, statistic_location=0.61528754, statistic_sign=1))
500 0.03 5 (0.9001133104970345, KstestResult(statistic=0.11932598004923373, pvalue=4.3035914102057145e-26, statistic_location=0.67538667, statistic_sign=-1), KstestResult(statistic=0.003816513632757146, pvalue=0.09035914960403713, statistic_location=0.62320995, statistic_sign=1))
500 0.03 6 (0.9005770767335171, KstestResult(statistic=0.19376467563310987, pvalue=2.445837738742191e-68, statistic_location=0.6718813, statistic_sign=-1), KstestRes

500 0.09 2 (0.8984163953110856, KstestResult(statistic=0.04089814499635425, pvalue=0.0019158280987845748, statistic_location=0.618322, statistic_sign=-1), KstestResult(statistic=0.0030887616828454423, pvalue=0.26236029696020235, statistic_location=0.6874866, statistic_sign=1))
500 0.09 3 (0.9018813088932037, KstestResult(statistic=0.0847523000629012, pvalue=2.2881361391926323e-13, statistic_location=0.62703335, statistic_sign=-1), KstestResult(statistic=0.0031852934445892878, pvalue=0.23083780997290515, statistic_location=0.6477317, statistic_sign=1))
500 0.09 4 (0.9004890910043123, KstestResult(statistic=0.160615776983497, pvalue=5.297344369127641e-47, statistic_location=0.57381207, statistic_sign=-1), KstestResult(statistic=0.0035796747056754397, pvalue=0.13109317435669354, statistic_location=0.6759265, statistic_sign=1))
500 0.09 5 (0.8993145160777261, KstestResult(statistic=0.26815795779893475, pvalue=2.903070348923322e-131, statistic_location=0.6762498, statistic_sign=-1), KstestR